# Simple Neural Networks with TensorFlow 2

In this lab, we are going to build several neural networks: a simple one at first (with no hidden layer), then a multilayer perceptron (several hidden layers). We will use the MNIST dataset to test how these networks learn in real conditions.

MNIST is the *Hello World!* dataset of machine learning. It comprises 28 $\times$ 28 grayscale images. These images picture digits (0 to 9). Each image comes with a label giving the character the image is supposed to represent.

## Libraries Imports

TensorFlow and the MNIST dataset

In [ ]:
import collections.abc
import functools
import itertools
import random
import typing

import numpy
import matplotlib
import matplotlib.pyplot as plt
import seaborn
import tensorflow as tf
import keras

[`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) & [`seaborn`](https://seaborn.pydata.org/) are basic Machine Learning libraries that we are going to use to visualize things and to perform various simple operations on matrices.

## Getting the Data

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Visualizing the Data

Use the following cell to analyze the 'mnist' object. You can use completion to find out its attributes and methods.

Having in mind that we are going to employ supervised Machine Learning, try to formulate clearly what are going to be our training and test sets, and to display a few examples.

To guide your exploration, try to answer the following questions:

- How many images can you find in total in this dataset?
- What are the shapes of the different arrays that you can find? And for labels ?
- Are the classes balanced?

In [ ]:
# Your code here

### Solution

In [ ]:
example = X_train[0]
example_label = y_train[0]

print(f"Examples shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")

plt.imshow(example, cmap="gray_r")
plt.title(f"First example of the dataset ({example_label})")
plt.show()

seaborn.countplot(x=y_train)
plt.title("Counts of the different classes (digits)")
plt.ylabel("Count")
plt.xlabel("Digit")
plt.show()

## Displaying a few examples

Display 25 examples, drawn randomly from the train set. Also display the corresponding labels.

You can use the following:
- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.imshow.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Using subplots to create a 5x5 grid
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Your code here

# ax[2, 4].imshow(... , cmap="gray_r")
# ax[2, 4].set_title("Example n (label)")

### Solution

In [ ]:
# Using subplots to create a 5x5 grid
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Draw 25 indices randomly
random_indexes = numpy.random.choice(X_train.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = X_train[img_index]
    label = y_train[img_index]

    # Use imshow to display the images
    ax[i, j].imshow(image, cmap='gray_r')
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Data transformation

We need to apply a few transformations to this data:

1. We could keep the original shapes of the numpy arrays `(_, 28, 28)` but it will be easier to work with tensors of shape `(_, 28²)` where `_` is the original number of examples.
2. We want to use the $[0, 1]$ range rather than values between $0$ and $255$ for our images. Transform the inputs accordingly.

Useful function: [`numpy.ndarray.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html)

*Apply both transformations to `X_train` and `X_test`.*

In [ ]:
# Your code here

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# We want to "flatten" X so that our network's input is a vector
# of size input_dim
X_train = X_train.reshape(X_train.shape[0], input_dim)
# equivalent syntax
X_train = X_train.reshape(-1, input_dim)

X_test = X_test.reshape(X_test.shape[0], input_dim)
# equivalent syntax
X_test = X_test.reshape(-1, input_dim)

# Transform to [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

## Building a shallow neural network

### Model variables

In a shallow network, $y = \sigma(XW +b)$, where $\sigma$ is an activation function that suits the problem. Here, we will pick the softmax function, to obtain a probability distribution over our multiple classes.

Note on variable names: in the slides, we use $\theta$ to denote all the weights: the regular ones that are the input coefficients — and the biases). Here, we will name the former `W` and the latter `b`.

*Fill in the `create_simple_nn_weights` function to initialize `W` & `b` as [TensorFlow variables](https://www.tensorflow.org/guide/variable) with the [`tensorflow.random.normal`](https://www.tensorflow.org/api_docs/python/tf/random/normal) function.*

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[tf.Variable, tf.Variable]:
  W = None  # Your code here
  b = None  # Your code here
  return W, b

#### Solution

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[tf.Variable, tf.Variable]:
  return (tf.Variable(tf.random.normal((n_inputs, n_outputs)), name="W"),
          tf.Variable(tf.random.normal((n_outputs,)), name="b"))

### Model definition

We are going to implement our model class.

*Tweak the following code so that `y_pred` is the correct output of our shallow network: $\text{softmax}(XW+b)$. We will use the [softmax log](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax) instead of the regular softmax for numerical stability.*

In [ ]:
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    y_pred = X  # Your code here
    return y_pred


simple_nn = SimpleNN()
simple_nn(X_train)

#### Solution

In [ ]:
# Create the model
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    return tf.nn.log_softmax(X @ self.W + self.b)


simple_nn = SimpleNN()
simple_nn(X_train)

### Computing the loss function

We're going to use a standard loss function for multi-class classification: cross entropy. To do so, you can call [`tensorflow.keras.losses.sparse_categorical_crossentropy`](https://www.tensorflow.org/api_docs/python/tf/keras/losses/sparse_categorical_crossentropy). Moreover, we want to be able to interpret this value directly, so the `categorical_crossentropy` function must return an aggregated value (the mean), which the Keras function doesn't do.

You can use [`tensorflow.math.reduce_mean`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean) for that.

*Complete the `categorical_crossentropy` function.*

In [ ]:
def categorical_crossentropy(y: numpy.ndarray, y_pred: tf.Tensor) -> tf.Tensor:
  loss = None  # Your code here
  return loss


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

#### Solution

In [ ]:
def categorical_crossentropy(y: numpy.ndarray, y_pred: tf.Tensor) -> tf.Tensor:
  cross_entropy = keras.losses.sparse_categorical_crossentropy(
      y, y_pred, from_logits=True)
  return tf.reduce_mean(cross_entropy)


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

### Metrics

To know whether a model is learning correctly, it's important to measure its performance. In this lab, we're going to use the simplest metric, which is also one of the most informative: accuracy. It's simply the number of correct predictions over the total number of predictions.

*Use [numpy.argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) and numpy array comparisons to complete the accuracy function.*

In [ ]:
def accuracy(y: numpy.ndarray, y_pred: tf.Tensor) -> float:
  predictions = None  # Your code here
  return None


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

#### Solution

In [ ]:
def accuracy(y: numpy.ndarray, y_pred: tf.Tensor) -> float:
  argmaxed = tf.math.argmax(y_pred, axis=-1)
  n_equal = tf.math.count_nonzero(argmaxed == y)
  return (n_equal / y.shape[0]).numpy()


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

## TensorFlow dataset creation

- *Use [`tensorflow.data.Dataset.from_tensor_slices`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices) to implement the `create_dataset` function, that creates a TensorFlow dataset from the `X_train` & `y_train` NumPy arrays. We will use batches of the following shape: `(X_batch, y_batch)`.*
- *Use the [`batch`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch) function to gather examples into batches.*
- *Shuffle the obtained dataset with the [`shuffle`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#shuffle) function, make sure that the dataset will be shuffled at each iteration.*

In [ ]:
def create_dataset(X: tf.Tensor, y: numpy.ndarray, batch_size: int = 4000
                   ) -> tf.data.Dataset:
  pass  # Your code here

### Solution

In [ ]:
def create_dataset(X: tf.Tensor, y: numpy.ndarray, batch_size: int = 4000
                   ) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices((X, y))
  dataset = dataset.batch(batch_size)
  dataset = dataset.shuffle(10_000, reshuffle_each_iteration=True)
  return dataset

## Training

Here is an *almost* complete training loop.

*Fill in the blanks to perform the parameters update. You can use [`tensorflow.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape) to achieve that.*

In [ ]:
LossFunction = collections.abc.Callable[[numpy.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = collections.abc.Callable[[tf.Module], tf.Tensor]


def train(model: typing.Any,
          X_train: tf.Tensor = X_train,
          y_train: numpy.ndarray = y_train,
          X_val: tf.Tensor = X_test,
          y_val: numpy.ndarray = y_test,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: typing.Optional[RegularizationFunction] = None,
         ) -> tuple[list[float], ...]:
  y_train_pred = model(X_train)
  y_val_pred = model(X_val)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [loss(y_train, y_train_pred)]
  val_losses = [loss(y_val, y_val_pred)]

  print(f"Initial metrics: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train.as_numpy_iterator():

      pass # Your code here

    # Compute and display the evaluation metric on the validation set
    # every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      y_train_pred = model(X_train)
      y_val_pred = model(X_val)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(loss(y_train, y_train_pred))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(loss(y_val, y_val_pred))
      print(f"Epoch {e + 1} metrics: acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

### Solution

In [ ]:
LossFunction = collections.abc.Callable[[numpy.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = collections.abc.Callable[[tf.Module], tf.Tensor]


def train(model: typing.Any,
          X_train: tf.Tensor = X_train,
          y_train: numpy.ndarray = y_train,
          X_val: tf.Tensor = X_test,
          y_val: numpy.ndarray = y_test,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: typing.Optional[RegularizationFunction] = None,
         ) -> tuple[list[float], ...]:
  y_train_pred = model(X_train)
  y_val_pred = model(X_val)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [loss(y_train, y_train_pred)]
  val_losses = [loss(y_val, y_val_pred)]

  print(f"Initial metrics: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train.as_numpy_iterator():

      # Compute the loss while recording the operations
      with tf.GradientTape() as tape:
        y_train_batch_pred = model(X_train_batch)
        loss_scalar = loss(y_train_batch, y_train_batch_pred)
        if regularization is not None:
          loss_scalar += regularization(model)

      # Use the recording to automatically compute the
      # gradient
      gradients = tape.gradient(loss_scalar, model.trainable_variables)

      # Go through the variables one by one to update them following the
      # rule: new_value = old_value - learning_rate * gradient
      for gradient, variable in zip(gradients, model.trainable_variables):
        variable.assign_sub(gradient * learning_rate)

    # Compute and display the evaluation metric on the validation set
    # every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      y_train_pred = model(X_train)
      y_val_pred = model(X_val)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(loss(y_train, y_train_pred))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(loss(y_val, y_val_pred))
      print(f"Epoch {e + 1} metrics: acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

## Adding depth

To implement deep neural networks, we “simply” need to add weight matrices and bias vectors for each layer in our weight creation function, and use them in our model. Here is the tweaked weight creation function:

In [ ]:
def create_mlp_weights(n_inputs: int,
                       n_outputs: int,
                       hidden_layer_sizes: list[int] = [512],
                      ) -> list[dict[str, tf.Variable]]:
  variables = []

  # Keep track of the last output size: it will be the new
  # input size. Starts at n_inputs
  last_size = n_inputs

  for i, hidden_layer_size in enumerate(hidden_layer_sizes, 1):
    # Randomly initialize a weight matrix and a bias vector
    W = tf.random.normal((last_size, hidden_layer_size))
    b = tf.zeros((hidden_layer_size,))

    # Add these two variables to a dictionary, then to the list of
    # our variables
    variables.append(dict(W=tf.Variable(W), b=tf.Variable(b)))

    # Update the last output size used
    last_size = hidden_layer_size

  # The last layer is special: its output size is always n_outputs,
  # so we handle it separately
  W = tf.random.normal((last_size, n_outputs))
  b = tf.zeros((n_outputs,))
  variables.append(dict(W=tf.Variable(W), b=tf.Variable(b)))
  return variables

We now compute the output of the model iteratively, one layer at a time. Each layer takes as input the output of the previous layer. For simplicity, you can pick the activation function of the hidden layers yourself.

*Complete the `MLP.__call__` function to compute the output of a deep neural network. Each variable is available as `self.weights[layer_number][variable_name]`. For example, to access the `W` matrix of the first layer: `self.weights[0]["W"]`.*

In [ ]:
class MLP(tf.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Your code here
    return X

### Solution

In [ ]:
class MLP(tf.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    current = X
    for layer_weights in self.weights:
      # Compute the non-activated output of the layer of neurons
      current = current @ layer_weights["W"] + layer_weights["b"]
      # Activation if we're not in the output layer
      if layer_weights is not self.weights[-1]:
        current = tf.nn.tanh(current)
    return tf.nn.log_softmax(current)

## Training the deep neural network

In [ ]:
_ = train(MLP(), learning_rate=0.5, epochs=100)

*What do you notice when training a deep neural network with 512 hidden neurons ?*

Your answer here

### Solution

The network learns less well than the previous network, which had no hidden layer: it has a nearly perfect loss on the training set but makes almost 50% more errors on validation. This is a case of overfitting.

## Regularization

Regularization is a good way to fight the phenomenon we observe when training our network with a hidden layer.

To set it up, we add a term to the loss function that penalizes large weight values.

*Modify the function below to compute the L2 penalty. It's equal to the sum of the squares of the parameters.*

In [ ]:
def l2(lambda_coefficient: float) -> collections.abc.Callable[[tf.Module], tf.Tensor]:
  def worker(model: tf.Module) -> tf.Tensor:
    weights_norm = tf.constant(0, dtype="float32")
    # Your code here
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=0.3,
      regularization=l2(0.3))

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> collections.abc.Callable[[tf.Module], tf.Tensor]:
  def worker(model: tf.Module) -> tf.Tensor:
    weights_norm = tf.constant(0, dtype="float32")
    for variable in model.variables:
      weights_norm += tf.reduce_sum(tf.math.square(variable))
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=3e-1,
      regularization=l2(3e-3))

## Hyper-parameters search

We are now going to implement a basic hyper-parameters search procedure. Use [`random.choice`](https://docs.python.org/3/library/random.html#random.choice) to sample the learning rate and the number of hidden neurons in the hidden layer and return a list of the tried parameters and a metric for each try.

In [ ]:
def search_hyperparameters(
    X_train: tf.Tensor = X_train,
    y_train: numpy.ndarray = y_train,
    X_val: tf.Tensor = X_test,
    y_val: numpy.ndarray = y_test,
    learning_rates: list[int] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[dict[str, typing.Any], float]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    pass
  return params, val_accs


search_hyperparameters()

### Solution

In [ ]:
def search_hyperparameters(
    X_train: tf.Tensor = X_train,
    y_train: numpy.ndarray = y_train,
    X_val: tf.Tensor = X_test,
    y_val: numpy.ndarray = y_test,
    learning_rates: list[int] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[dict[str, typing.Any], float]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    learning_rate = random.choice(learning_rates)
    n_hidden_units = random.choice(ns_hidden_units)
    mlp = MLP(hidden_layer_sizes=[n_hidden_units])
    _, val_acc_epochs, _, _ = train(mlp, X_train, y_train, X_val, y_val,
                                    learning_rate=learning_rate, epochs=50)
    val_accs.append(val_acc_epochs[-1])
    params.append(dict(learning_rate=learning_rate,
                       n_hidden_units=n_hidden_units))
  return params, val_accs


search_hyperparameters()